In [1]:
import h5py
import numpy as np
import pandas as pd
from glob import glob

import matplotlib.pyplot as plt

filename = 'DB/test/rgb_30d_test.h5' 

with h5py.File(filename, 'r') as f:
    print("Keys:", list(f.keys()))
    permnos = f['permnos'][:]
    dates = f['dates'][:]
    image = f['images'][-1]
    # Count & indices for a specific permno

Keys: ['dates', 'images', 'permnos']


In [67]:
temp = pd.read_csv("DB/crsp_ver2_price_2603.csv")

C:\Users\USER\AppData\Local\Temp\ipykernel_19348\3284075337.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  temp = pd.read_csv("DB/crsp_ver2_price_2603.csv")


In [84]:
temp[temp['DlyCalDt'] >='2025-00-00'][['PERMNO','IssuerNm']].drop_duplicates().reset_index(drop=True).to_csv('DB/test_sample_indicator.csv')

In [2]:
info_table = pd.DataFrame(permnos,dates).reset_index().copy()
info_table.columns = ['date','PERMNO']
info_table = info_table[info_table['date']>20250000].copy()
info_table = info_table.reset_index()
info_table = info_table.rename(columns={'index': 'num'})

In [ ]:
paths = glob("experiments/ViT_B32_30d/enc2_batch1024_rs2ratio0.1_lr0.0001_wd0.05/*/valid_losses*")

results = []
for path in paths:

    seed = path.split("seed")[1].split("\\")[0].split("/")[0]
    
    with open(path, 'r') as f:
        losses = [float(line.strip()) for line in f if line.strip()]
    
    best_loss = min(losses)
    best_epoch = losses.index(best_loss) + 1  # 1-indexed
    
    results.append({
        'seed': seed,
        'best_loss': best_loss,
        'best_epoch': best_epoch,
        'total_epochs': len(losses)
    })

results.sort(key=lambda x: x['best_loss'])

print(f"{'Seed':<10} {'Best Loss':<15} {'Best Epoch':<12} {'Total Epochs'}")
print("-" * 50)
for r in results:
    marker = " ★" if r == results[0] else ""
    print(f"seed{r['seed']:<6} {r['best_loss']:<15.6f} {r['best_epoch']:<12} {r['total_epochs']}{marker}")

Seed       Best Loss       Best Epoch   Total Epochs
--------------------------------------------------
seed92     0.682018        68           78 ★
seed60     0.682164        69           79
seed14     0.682245        69           79
seed51     0.682274        66           76
seed71     0.682288        69           79


In [23]:
prob_lst = []
for csv_path in glob("monthly_prediction/b32_30d_enc2/*"):
    temp = pd.read_csv(csv_path)
    date = int(csv_path.split('\\')[-1].split('.csv')[0].replace('-',''))

    if date < 20250000:
        continue

    temp = temp[['PERMNO','seed92']].copy()
    temp['date'] = date
    prob_lst.append(temp)

prob_df = pd.concat(prob_lst,axis=0)

df = prob_df.merge(info_table, left_on=["PERMNO", "date"], right_on=["PERMNO", "date"])

In [24]:
cap = pd.read_pickle("DB/cap.pkl")
cap = cap.reset_index()
cap = cap.rename(columns={'index': 'date'})
cap['date'] = cap['date'].dt.strftime('%Y%m%d').astype(int)
cap = cap.melt(id_vars='date', var_name='PERMNO', value_name='cap')
cap = cap.dropna(subset=['cap'])
cap['PERMNO'] = cap['PERMNO'].astype(int)

df = df.merge(cap, left_on=["PERMNO", "date"], right_on=["PERMNO", "date"])

In [25]:
final_ret_df = pd.read_pickle("DB/ret_w_delret.pkl")
not_na_mask = final_ret_df.notna()
first_valid_mask = not_na_mask & (not_na_mask.cumsum() == 1)
final_ret_df[first_valid_mask] = 0.0
cls = (final_ret_df + 1).cumprod()

In [26]:
ret_table = []
for date in sorted(pd.to_datetime(df['date'].unique().astype('str'))):
    temp = cls.loc[date:].iloc[1:22]
    temp = temp[temp.iloc[0].dropna().index].ffill()
    temp = pd.DataFrame((temp.iloc[-1]/temp.iloc[0]) - 1)
    temp.columns = ["RET"]
    temp['date'] = int(str(date).split(' ')[0].replace('-',''))
    temp = temp.reset_index()
    temp = temp.rename(columns={'index': 'PERMNO'})
    ret_table.append(temp)

ret_df = pd.concat(ret_table,axis=0)

In [27]:
df = df.merge(ret_df, left_on=["PERMNO", "date"], right_on=["PERMNO", "date"])

In [28]:
df['prob_rank'] = df.groupby('date')['seed92'].transform(
    lambda x: x.rank(pct=True)
)
df['cap_rank'] = df.groupby('date')['cap'].transform(
    lambda x: x.rank(pct=True)
)

In [30]:
df['ret_rank'] = df.groupby('date')['RET'].transform(
    lambda x: x.rank(pct=True)
)

In [32]:
df = df[df['date']!=20251219]

In [39]:
df = df[(df['prob_rank'] <= 0.1) | (0.9<=df['prob_rank'])]

df = df[df['cap_rank']>=0.1]

In [40]:
df = df.sort_values(by='num').reset_index(drop=True)

In [ ]:
df.to_csv("DB/xai_candidate_table.csv")

In [50]:
df

,PERMNO,seed92,date,num,cap,RET,prob_rank,cap_rank,ret_rank
0,10026,0.571976,20250502,951,2.515142e+06,-0.126592,0.902353,0.678117,0.104409
1,10026,0.592976,20250925,956,1.854716e+06,-0.047903,0.955367,0.616101,0.285871
2,10028,0.377655,20250403,1255,1.754528e+05,0.104433,0.035878,0.325870,0.560291
3,10028,0.418311,20250730,1259,1.481715e+05,0.354243,0.080447,0.278975,0.939733
4,10032,0.594389,20250306,1569,3.361883e+06,-0.100353,0.967292,0.718231,0.608579
...,...,...,...,...,...,...,...,...,...
7150,93436,0.610570,20250403,1306169,8.606416e+08,0.170531,0.979094,0.998381,0.737254
7151,93436,0.612918,20250502,1306170,9.250908e+08,0.228399,0.987152,0.998107,0.878280
7152,93436,0.589052,20250730,1306173,1.029047e+09,0.122328,0.957731,0.997818,0.743660
7153,93436,0.574171,20250827,1306174,1.127617e+09,0.272905,0.903578,0.997815,0.927889


In [31]:
temp = pd.read_csv("DB/xai_candidate_table.csv",index_col=0)

In [ ]:
down_examples = [309361,307154,226771,
                 242320,333498,342613,
                 256170,305222,804086]

In [35]:
down_temp = temp[temp['num'].isin(down_examples)].copy()
down_temp

,PERMNO,seed92,date,num,cap,RET,prob_rank,cap_rank,ret_rank
1552,17344,0.398085,20250701,226771,66204.79,-0.101064,0.049837,0.198257,0.199891
1737,18137,0.392744,20250306,242320,428666.22,-0.339450,0.078552,0.441287,0.087936
1937,18840,0.423816,20250925,256170,54449.64,-0.080000,0.079682,0.161829,0.182366
2841,21350,0.390447,20251023,305222,493681.44,-0.207802,0.080636,0.423204,0.239440
2886,21547,0.432895,20251120,307154,867358.80,-0.403923,0.099973,0.516891,0.017578
2947,21625,0.365174,20250306,309361,103092.39,-0.322222,0.043566,0.257909,0.103217
3475,23052,0.360292,20250827,333498,44803.02,-0.372256,0.024310,0.148593,0.013111
3684,23860,0.341418,20250502,342613,163980.32,-0.115787,0.015418,0.307547,0.117392
5631,83112,0.383670,20250502,804086,1292648.28,-0.045159,0.042196,0.594807,0.224777


In [43]:
def fmt_prob(p, r):
    return f"{p:.3f} ({r:.3f})"

def fmt_ret(r, rk):
    return f"{r*100:.1f}% ({rk:.3f})"

In [ ]:
down_temp['prob_up'] = down_temp['seed92']

down_temp['Prob (Rank)'] = down_temp['prob_up'].round(3).astype(str) + ' (' + down_temp['prob_rank'].round(3).astype(str) + ')'
down_temp['RET (Rank)']  = (down_temp['RET']*100).round(1).astype(str) + '% (' + down_temp['ret_rank'].round(3).astype(str) + ')'
down_temp['Cap Rank']    = down_temp['cap_rank'].round(3)

down_temp['Prob (Rank)'] = down_temp.apply(lambda x: fmt_prob(x['prob_up'], x['prob_rank']), axis=1)
down_temp['RET (Rank)']  = down_temp.apply(lambda x: fmt_ret(x['RET'], x['ret_rank']), axis=1)
down_temp['Cap Rank']    = down_temp['cap_rank'].apply(lambda x: f"{x:.3f}")

result = down_temp[['PERMNO', 'date', 'Prob (Rank)', 'RET (Rank)', 'Cap Rank']]
print(result.to_latex(index=False))

\begin{tabular}{rrlll}
\toprule
PERMNO & date & Prob (Rank) & RET (Rank) & Cap Rank \\
\midrule
17344 & 20250701 & 0.398 (0.050) & -10.1% (0.200) & 0.198 \\
18137 & 20250306 & 0.393 (0.079) & -33.9% (0.088) & 0.441 \\
18840 & 20250925 & 0.424 (0.080) & -8.0% (0.182) & 0.162 \\
21350 & 20251023 & 0.390 (0.081) & -20.8% (0.239) & 0.423 \\
21547 & 20251120 & 0.433 (0.100) & -40.4% (0.018) & 0.517 \\
21625 & 20250306 & 0.365 (0.044) & -32.2% (0.103) & 0.258 \\
23052 & 20250827 & 0.360 (0.024) & -37.2% (0.013) & 0.149 \\
23860 & 20250502 & 0.341 (0.015) & -11.6% (0.117) & 0.308 \\
83112 & 20250502 & 0.384 (0.042) & -4.5% (0.225) & 0.595 \\
\bottomrule
\end{tabular}



In [40]:
up_examples = [148848,602322,861269,
               106465,153489,260595,
               89640,355142,488310]

In [41]:
up_temp = temp[temp['num'].isin(up_examples)].copy()

In [42]:
up_temp

,PERMNO,seed92,date,num,cap,RET,prob_rank,cap_rank,ret_rank
354,13188,0.576661,20250306,89640,6.459846e+06,0.005131,0.924799,0.805630,0.893566
465,13721,0.584419,20250827,106465,3.675178e+08,0.061033,0.934581,0.993444,0.727124
778,14714,0.616243,20250502,148848,1.142871e+08,0.045586,0.989721,0.977549,0.556127
823,14823,0.581882,20250730,153489,2.016306e+06,0.081175,0.934824,0.638942,0.620671
2039,19076,0.588963,20250701,260595,1.130060e+07,0.163203,0.923475,0.854031,0.895153
3987,25252,0.580248,20250827,355142,1.818089e+06,0.058261,0.924884,0.609669,0.722480
4668,58819,0.582120,20250205,488310,1.533221e+07,0.049547,0.953672,0.878505,0.879306
5044,76709,0.574977,20250502,602322,3.795114e+07,0.106848,0.913714,0.937517,0.729510
5760,84403,0.624494,20250502,861269,1.153843e+07,0.178531,0.995131,0.865567,0.831755


In [ ]:
up_temp['prob_up'] = up_temp['seed92']

up_temp['Prob (Rank)'] = up_temp['prob_up'].round(3).astype(str) + ' (' + up_temp['prob_rank'].round(3).astype(str) + ')'
up_temp['RET (Rank)']  = (up_temp['RET']*100).round(1).astype(str) + '% (' + up_temp['ret_rank'].round(3).astype(str) + ')'
up_temp['Cap Rank']    = up_temp['cap_rank'].round(3)

up_temp['Prob (Rank)'] = up_temp.apply(lambda x: fmt_prob(x['prob_up'], x['prob_rank']), axis=1)
up_temp['RET (Rank)']  = up_temp.apply(lambda x: fmt_ret(x['RET'], x['ret_rank']), axis=1)
up_temp['Cap Rank']    = up_temp['cap_rank'].apply(lambda x: f"{x:.3f}")

result = up_temp[['PERMNO', 'date', 'Prob (Rank)', 'RET (Rank)', 'Cap Rank']]
result

,PERMNO,date,Prob (Rank),RET (Rank),Cap Rank
354,13188,20250306,0.577 (0.925),0.5% (0.894),0.806
465,13721,20250827,0.584 (0.935),6.1% (0.727),0.993
778,14714,20250502,0.616 (0.990),4.6% (0.556),0.978
823,14823,20250730,0.582 (0.935),8.1% (0.621),0.639
2039,19076,20250701,0.589 (0.923),16.3% (0.895),0.854
3987,25252,20250827,0.580 (0.925),5.8% (0.722),0.610
4668,58819,20250205,0.582 (0.954),5.0% (0.879),0.879
5044,76709,20250502,0.575 (0.914),10.7% (0.730),0.938
5760,84403,20250502,0.624 (0.995),17.9% (0.832),0.866


In [54]:
print(result.to_latex())

\begin{tabular}{lrrlll}
\toprule
 & PERMNO & date & Prob (Rank) & RET (Rank) & Cap Rank \\
\midrule
354 & 13188 & 20250306 & 0.577 (0.925) & 0.5% (0.894) & 0.806 \\
465 & 13721 & 20250827 & 0.584 (0.935) & 6.1% (0.727) & 0.993 \\
778 & 14714 & 20250502 & 0.616 (0.990) & 4.6% (0.556) & 0.978 \\
823 & 14823 & 20250730 & 0.582 (0.935) & 8.1% (0.621) & 0.639 \\
2039 & 19076 & 20250701 & 0.589 (0.923) & 16.3% (0.895) & 0.854 \\
3987 & 25252 & 20250827 & 0.580 (0.925) & 5.8% (0.722) & 0.610 \\
4668 & 58819 & 20250205 & 0.582 (0.954) & 5.0% (0.879) & 0.879 \\
5044 & 76709 & 20250502 & 0.575 (0.914) & 10.7% (0.730) & 0.938 \\
5760 & 84403 & 20250502 & 0.624 (0.995) & 17.9% (0.832) & 0.866 \\
\bottomrule
\end{tabular}

